# Lesson 6.6: The Two-Sample t-Test

Compare exam scores of 30 students who studied with music and 30 who studied in silence, with Welch's and the pooled two-sample t-test.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A teacher wonders whether listening to music while studying hurts exam results. Sixty students were split at random into two groups of 30: one group studied with music, the other in silence, and then everyone sat the same exam. The silent group scored 5 points higher on average. Is that a real difference, or could two groups from the same population differ that much by luck?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Enter the two groups

H0: the true average scores of the music and silence groups are equal. H1: they differ. The scores below are built to match the lesson: music averages 74 (SD 8), silence 79 (SD 7), 30 students each. The students in each group are different people, so the groups are independent.

In [1]:
import numpy as np  # Load NumPy for arrays and arithmetic, under the short name np.
from scipy import stats  # Load SciPy's statistics tools, which hold the two-sample t-test.

music = np.array([77.91, 82.44, 77.77, 62.24, 83.23, 78.87, 69.52, 80.15, 78.09, 77.42, 74.9, 79.82, 67.63, 73.08, 70.05, 80.32, 75.01, 71.85, 67.2, 72.18, 74.71, 72.01, 86.93, 84.2, 48.86, 56.67, 72.97, 70.62, 76.66, 76.69])  # Store the 30 music-group scores as an array, ready for calculations.
silence = np.array([72.97, 64.46, 82.5, 83.47, 76.26, 80.89, 90.07, 85.63, 83.21, 83.08, 72.47, 84.97, 67.1, 76.11, 75.39, 66.56, 77.62, 89.61, 76.05, 70.12, 77.75, 84.22, 79.7, 78.36, 68.74, 81.8, 81.7, 87.55, 88.56, 83.08])  # Store the 30 silence-group scores the same way.

for name, g in [('music', music), ('silence', silence)]:  # Loop over both groups with their names, so we describe each group with the same code instead of repeating it.
    print(f'{name}: n = {len(g)}, mean = {g.mean():.2f}, sd = {g.std(ddof=1):.2f}')  # For each group print its size, mean and sample SD (ddof=1), so we can see centre and spread before testing.

music: n = 30, mean = 74.00, sd = 8.00
silence: n = 30, mean = 79.00, sd = 7.00


**What this means.** Music averages 74.00 and silence 79.00, a 5-point gap, with SDs of 8.00 and 7.00. The spreads are similar, which matters for choosing the test. The gap is real in these 60 students; the test will ask whether chance alone could produce it.

## Step 2: Welch's t-test (the safe default)

equal_var=False gives Welch's test, which does not assume the two groups have the same spread. The degrees of freedom are not a whole number because they are adjusted for the unequal spreads.

In [2]:
welch = stats.ttest_ind(music, silence, equal_var=False)  # ttest_ind compares two independent groups. equal_var=False asks for Welch's test, which does not assume the groups have the same spread, so it is the safer default.

print(f't  = {welch.statistic:.3f}')  # Print t, the gap between the means in standard-error units. It is negative because music is lower than silence.
print(f'df = {welch.df:.1f}')  # Print the degrees of freedom. They are not a whole number because Welch's method adjusts them for unequal spreads.
print(f'p  = {welch.pvalue:.4f}')  # Print the two-sided p-value to 4 decimals.

t  = -2.576
df = 57.0
p  = 0.0126


**What this means.** Welch's t = -2.576 with df = 57.0 and p = 0.0126. The negative sign only says music is lower. If both groups really had the same mean, a gap this big would appear about 1.3% of the time. That is below 0.05, so we reject H0. It does not say there is a 1.3% chance that H0 is true.

## Step 3: The pooled t-test

equal_var=True blends the two variances into one. Here the group sizes are equal and the spreads are similar, so the two tests agree closely. They would not agree if the smaller group were also the noisier one.

In [3]:
pooled = stats.ttest_ind(music, silence, equal_var=True)  # equal_var=True gives the pooled test, which blends the two variances into one and assumes they are equal.

print(f't  = {pooled.statistic:.3f}')  # Print t; the pooled and Welch versions are expected to be nearly the same here.
print(f'df = {pooled.df:.0f}')  # Print the degrees of freedom: n1 + n2 - 2 = 58, a whole number for the pooled test.
print(f'p  = {pooled.pvalue:.4f}')  # Print the p-value to 4 decimals.

t  = -2.576
df = 58
p  = 0.0126


**What this means.** The pooled test gives t = -2.576, df = 58 and p = 0.0126, almost the same as Welch. The groups have equal size and similar spreads, so the two agree. They would not if the smaller group were the noisier one. Because Welch stays valid either way, it is the sensible default.

## Step 4: Confidence interval for the difference

The test says whether the gap is real, and the interval says how big it might be. It is for music minus silence, so a fully negative interval means music scores are lower.

In [4]:
ci = welch.confidence_interval(confidence_level=0.95)  # Ask the Welch result for a 95% confidence interval for the difference in means (music minus silence).
diff = music.mean() - silence.mean()  # Compute the difference in sample means directly, so we can see the size of the gap.

print(f'difference (music - silence) = {diff:.2f}')  # Print the observed difference, rounded to 2 decimals.
print(f'95% CI: ({ci.low:.2f}, {ci.high:.2f})')  # Print the interval ends: the range of true differences that fit the data.
print('interval contains 0?', ci.low <= 0 <= ci.high)  # Check whether 0 (no difference) lies inside the interval. This prints True or False.

difference (music - silence) = -5.00
95% CI: (-8.89, -1.11)
interval contains 0? False


**What this means.** Music minus silence is -5.00. The 95% interval, -8.89 to -1.11, excludes 0, which agrees with p < 0.05. The true gap is plausibly anywhere from about 1 to 9 points in silence's favour. The p-value says the gap is probably not zero; the interval says how big it might be.

## So, what is the answer?

Students who studied in silence scored about 5 points higher than those who studied with music (Welch t = -2.576, p = 0.0126, 95% CI -8.89 to -1.11). Because students were assigned at random, music is a reasonable explanation. Caution: significant does not mean large, the interval still allows a gap of only 1 point, and one exam in one class may not carry over elsewhere.

## Now you try

A food company compares two couriers, A and B, to see whether their average delivery times (in minutes) differ. They time 10 deliveries for each courier on different days. Run Welch's t-test and a 95% confidence interval for the difference, then compare with the pooled test.

**Data:** a = [27.2, 31.8, 37.3, 28.5, 31.3, 32.7, 32.4, 24.6, 32.5, 40.2]
b = [13.1, 34.7, 28.1, 21.2, 45.0, 33.9, 16.2, 27.7, 32.2, 25.3]

1. Find the mean and SD of each courier. Do the spreads look similar?
2. Run Welch's t-test and report t, df and p.
3. Give the 95% confidence interval for the difference in means (A minus B).
4. Compare with the pooled p-value and say what the result does not prove.

*Hint:* Use stats.ttest_ind(a, b, equal_var=False) for Welch, .confidence_interval(0.95) for the interval, and equal_var=True for the pooled test.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np
from scipy import stats

a = np.array([27.2, 31.8, 37.3, 28.5, 31.3, 32.7, 32.4, 24.6, 32.5, 40.2])
b = np.array([13.1, 34.7, 28.1, 21.2, 45.0, 33.9, 16.2, 27.7, 32.2, 25.3])
for name, g in [('A', a), ('B', b)]:
    print(f'courier {name}: mean = {g.mean():.2f}, sd = {g.std(ddof=1):.2f}')
w = stats.ttest_ind(a, b, equal_var=False)
ci = w.confidence_interval(0.95)
print(f'Welch: t = {w.statistic:.3f}, df = {w.df:.1f}, p = {w.pvalue:.4f}')
print(f'diff (A - B) = {a.mean() - b.mean():.2f}, 95% CI ({ci.low:.2f}, {ci.high:.2f})')
print(f'pooled p = {stats.ttest_ind(a, b).pvalue:.4f}')

courier A: mean = 31.85, sd = 4.55
courier B: mean = 27.74, sd = 9.43
Welch: t = 1.241, df = 13.0, p = 0.2366
diff (A - B) = 4.11, 95% CI (-3.05, 11.27)
pooled p = 0.2305


**Answer.** Courier A averages 31.85 min (SD 4.55) and B 27.74 min (SD 9.43), so B is far more variable. Welch gives t = 1.241, df = 13.0, p = 0.2366, above 0.05, so we fail to reject H0. The difference of 4.11 min has a 95% interval of -3.05 to 11.27, which includes 0. The pooled p (0.2305) agrees. This does not prove the couriers are equal: 10 deliveries each is a small sample.

## The same question in R, SPSS and Stata

**R**

```r
music <- c(77.91, 82.44, 77.77, 62.24, 83.23, 78.87, 69.52, 80.15, 78.09, 77.42, 74.9, 79.82, 67.63, 73.08, 70.05, 80.32, 75.01, 71.85, 67.2, 72.18, 74.71, 72.01, 86.93, 84.2, 48.86, 56.67, 72.97, 70.62, 76.66, 76.69)  # Store the 30 music scores in a vector called music.
silence <- c(72.97, 64.46, 82.5, 83.47, 76.26, 80.89, 90.07, 85.63, 83.21, 83.08, 72.47, 84.97, 67.1, 76.11, 75.39, 66.56, 77.62, 89.61, 76.05, 70.12, 77.75, 84.22, 79.7, 78.36, 68.74, 81.8, 81.7, 87.55, 88.56, 83.08)  # Store the 30 silence scores in a vector called silence.

t.test(music, silence)                    # Welch (the R default)  # t.test on two vectors is Welch's test by default in R, so no extra option is needed. It prints t, df, p and a 95% interval for the difference.
t.test(music, silence, var.equal = TRUE)  # pooled  # var.equal = TRUE asks for the pooled test instead, which assumes equal variances.
```

How to read it: In the Welch output read t = -2.5762, df = 56.997 and p-value = 0.01261. The interval, -8.886 to -1.114, excludes 0. The pooled output is nearly identical (df = 58, p = 0.01256). Both are below 0.05, so we reject H0 either way.

**SPSS** (Analyze > Compare Means > Independent-Samples T Test)

```
DATA LIST FREE / group score.
BEGIN DATA
1 77.91 1 82.44 1 77.77 1 62.24 1 83.23 1 78.87 1 69.52 1 80.15 1 78.09 1 77.42
1 74.9 1 79.82 1 67.63 1 73.08 1 70.05 1 80.32 1 75.01 1 71.85 1 67.2 1 72.18
1 74.71 1 72.01 1 86.93 1 84.2 1 48.86 1 56.67 1 72.97 1 70.62 1 76.66 1 76.69
2 72.97 2 64.46 2 82.5 2 83.47 2 76.26 2 80.89 2 90.07 2 85.63 2 83.21 2 83.08
2 72.47 2 84.97 2 67.1 2 76.11 2 75.39 2 66.56 2 77.62 2 89.61 2 76.05 2 70.12
2 77.75 2 84.22 2 79.7 2 78.36 2 68.74 2 81.8 2 81.7 2 87.55 2 88.56 2 83.08
END DATA.
T-TEST GROUPS=group(1 2)
  /VARIABLES=score
  /CRITERIA=CI(.95).
```

How to read it: In the Independent Samples Test table, use the row 'Equal variances not assumed' for Welch: t = -2.576, df = 57.0, Sig. (2-tailed) = 0.013. The 'Equal variances assumed' row is the pooled test (df = 58, 0.013). The mean difference is -5.00 with 95% interval -8.89 to -1.11.

**Stata**

```
ttesti 30 74 8 30 79 7              // pooled (Stata's default)
ttesti 30 74 8 30 79 7, unequal     // Welch-type test, same df as R and Python
```

How to read it: Look at t = -2.5766 for the rounded lesson numbers and the two-sided line Pr(|T| > |t|) = 0.0126. The Welch version (unequal) shows the same t, with df about 57.0 and a nearly identical p of 0.0126. Both are below 0.05.